In [ ]:
%pip install -U "kagglehub[pandas-datasets]"


In [ ]:
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

np.random.seed(42)

file_path = 'training.1600000.processed.noemoticon.csv'

df_raw = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    'kazanova/sentiment140',
    file_path,
    pandas_kwargs={
        'encoding': 'latin-1',
        'header': None,
        'on_bad_lines': 'skip',
    },
)

df_raw.head()

In [ ]:
df = df_raw.copy()
df.columns = ['target', 'id', 'date', 'flag', 'user', 'text']
df['target'] = df['target'].replace(4,1)
df = df[['text', 'target']]
df.head()

In [ ]:
import re

def clean_tweet(text):
    text = text.lower()  # lowercase

    text = re.sub(r'http\S+', '', text)  # remove URLs
    text = re.sub(r'@\w+', '', text)     # remove mentions
    text = re.sub(r'#\w+', '', text)     # remove hashtags

    text = re.sub(r'[^a-z\s]', '', text) # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces

    return text

In [ ]:
df['clean_text'] = df['text'].apply(clean_tweet)
df[['text', 'clean_text']].head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
X_train_tfidf.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)

model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred = model.predict(X_test_tfidf)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import re

EMOTICONS = [
    (r'(?<!\w):-?\)(?!\w)', 'smile'),
    (r'(?<!\w):-?\((?!\w)', 'sad'),
    (r'(?<!\w):d(?!\w)',    'laugh'),
    (r'(?<!\w)<3(?!\w)',    'love'),
    (r'(?<!\w);-?\)(?!\w)', 'wink'),
]

def replace_emoticons(text):
    for pattern, word in EMOTICONS:
        text = re.sub(pattern, f' {word} ', text)
    return text

def clean_tweet_advanced(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' USER ', text)
    text = re.sub(r'#(\w+)', r' \1 ', text)
    text = replace_emoticons(text)          # ← replaces the 6 bare lines
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
df['clean_text_advanced'] = df['text'].apply(clean_tweet_advanced)
df[['text', 'clean_text', 'clean_text_advanced']].head(10)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df['clean_text_advanced']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## Member 2 Task List

- Standardize a stratified train/validation/test split for fair comparison.
- Train and tune a linear SVM baseline on the validation split.
- Report accuracy, precision, recall, F1, ROC-AUC, and confusion matrices.
- Build a noisy-tweet subset from the held-out test set.
- Compare Logistic Regression and SVM on the full test set and the noisy subset.
- Draft the evaluation, error-analysis, and success-criteria sections.


In [ ]:
from sklearn.model_selection import train_test_split

if 'clean_text_advanced' not in df.columns:
    raise ValueError('Run the advanced preprocessing cells before the Member 2 section.')

member2_df = df[['text', 'clean_text_advanced', 'target']].dropna().copy()

train_df, holdout_df = train_test_split(
    member2_df,
    test_size=0.2,
    random_state=42,
    stratify=member2_df['target']
)

val_df, test_df = train_test_split(
    holdout_df,
    test_size=0.5,
    random_state=42,
    stratify=holdout_df['target']
)

train_val_df = pd.concat([train_df, val_df], ignore_index=True)

print(f'Train size: {len(train_df):,}')
print(f'Validation size: {len(val_df):,}')
print(f'Test size: {len(test_df):,}')


In [ ]:
from time import perf_counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.svm import LinearSVC

EMOTICON_PATTERN = re.compile(r'(:\)|:-\)|:\(|:-\(|:d|<3|;\)|;\-\))', re.IGNORECASE)
EMOJI_PATTERN = re.compile('[\U0001F300-\U0001FAFF\u2600-\u27BF]+')
REPEATED_CHAR_PATTERN = re.compile(r'(.)\1{2,}')
SLANG_PATTERN = re.compile(r'\b(lol|omg|wtf|idk|smh|tbh|imo|btw|lmao|rofl|ikr|u|ur|ya|tho|gonna|wanna)\b', re.IGNORECASE)

def add_noise_flags(frame):
    tagged = frame.copy()
    tagged['has_url'] = tagged['text'].str.contains(r'http\S+|www\.\S+', regex=True, na=False)
    tagged['has_mention'] = tagged['text'].str.contains(r'@\w+', regex=True, na=False)
    tagged['has_hashtag'] = tagged['text'].str.contains(r'#\w+', regex=True, na=False)
    tagged['has_repeated_chars'] = tagged['text'].apply(lambda value: bool(REPEATED_CHAR_PATTERN.search(str(value))))
    tagged['has_emoticon'] = tagged['text'].apply(lambda value: bool(EMOTICON_PATTERN.search(str(value))))
    tagged['has_emoji'] = tagged['text'].apply(lambda value: bool(EMOJI_PATTERN.search(str(value))))
    tagged['has_slang'] = tagged['text'].apply(lambda value: bool(SLANG_PATTERN.search(str(value))))

    noise_columns = [
        'has_url',
        'has_mention',
        'has_hashtag',
        'has_repeated_chars',
        'has_emoticon',
        'has_emoji',
        'has_slang',
    ]
    tagged['is_noisy'] = tagged[noise_columns].any(axis=1)
    return tagged, noise_columns

def build_vectorizer(ngram_range):
    return TfidfVectorizer(
        max_features=5000,
        ngram_range=ngram_range
    )

def metric_row(y_true, y_pred, y_score):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_score),
    }

def evaluate_model(model, vectorizer, frame, text_column='clean_text_advanced'):
    start = perf_counter()
    X_eval = vectorizer.transform(frame[text_column])
    y_pred = model.predict(X_eval)
    prediction_time = perf_counter() - start

    score_start = perf_counter()
    if hasattr(model, 'decision_function'):
        y_score = model.decision_function(X_eval)
    else:
        y_score = model.predict_proba(X_eval)[:, 1]
    scoring_time = perf_counter() - score_start

    metrics = metric_row(frame['target'], y_pred, y_score)
    metrics['prediction_time_sec'] = prediction_time
    metrics['scoring_time_sec'] = scoring_time
    metrics['tweets_per_second'] = len(frame) / prediction_time if prediction_time > 0 else float('inf')
    return metrics, y_pred, y_score

test_df, noise_columns = add_noise_flags(test_df)
noisy_test_df = test_df[test_df['is_noisy']].copy()
noise_summary = (
    test_df[noise_columns + ['is_noisy']]
    .mean()
    .sort_values(ascending=False)
    .rename('share_of_test_set')
)

print(f'Noisy subset size: {len(noisy_test_df):,} / {len(test_df):,}')
noise_summary


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

svm_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('svm',   LinearSVC(dual=False, max_iter=4000))
])

param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'svm__C':             [0.5, 1.0, 2.0],
}

gs = GridSearchCV(svm_pipe, param_grid, cv=3, scoring='f1', n_jobs=-1, verbose=1)
gs.fit(train_df['clean_text_advanced'], train_df['target'])

print("Best params:", gs.best_params_)
print("Best val F1:", round(gs.best_score_, 4))

best_ngram_range = gs.best_params_['tfidf__ngram_range']
best_c           = gs.best_params_['svm__C']

In [ ]:
best_ngram_range = tuple(best_svm_config['ngram_range'])
best_c = float(best_svm_config['C'])

comparison_vectorizer = build_vectorizer(best_ngram_range)
X_train_val_tfidf = comparison_vectorizer.fit_transform(train_val_df['clean_text_advanced'])

lr_shared = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)

from sklearn.calibration import CalibratedClassifierCV

svm_best = CalibratedClassifierCV(
    LinearSVC(C=best_c, dual=False, max_iter=4000),
    cv=3
)

model_store = {
    'Logistic Regression': lr_shared,
    'Linear SVM': svm_best,
}

comparison_rows = []
prediction_store = {}
report_store = {}

for model_name, model in model_store.items():
    train_start = perf_counter()
    model.fit(X_train_val_tfidf, train_val_df['target'])
    train_time = perf_counter() - train_start

    test_metrics, test_pred, _ = evaluate_model(model, comparison_vectorizer, test_df)
    test_metrics.update({
        'model': model_name,
        'split': 'full_test',
        'train_time_sec': train_time,
    })
    comparison_rows.append(test_metrics)

    prediction_store[model_name] = test_pred
    report_store[model_name] = classification_report(test_df['target'], test_pred, digits=4)

    if len(noisy_test_df) > 0:
        noisy_metrics, _, _ = evaluate_model(model, comparison_vectorizer, noisy_test_df)
        noisy_metrics.update({
            'model': model_name,
            'split': 'noisy_test',
            'train_time_sec': train_time,
        })
        comparison_rows.append(noisy_metrics)

comparison_results = pd.DataFrame(comparison_rows)
comparison_results[['model', 'split', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'train_time_sec', 'prediction_time_sec', 'scoring_time_sec', 'tweets_per_second']]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for model_name, report in report_store.items():
    print(model_name)
    print(report)

fig, axes = plt.subplots(1, len(prediction_store), figsize=(12, 4))

if len(prediction_store) == 1:
    axes = [axes]

for ax, (model_name, predictions) in zip(axes, prediction_store.items()):
    cm = confusion_matrix(test_df['target'], predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(model_name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()


In [ ]:
import joblib

joblib.dump(comparison_vectorizer, 'vectorizer.pkl')
joblib.dump(svm_best,              'svm_model.pkl')
joblib.dump(lr_shared,             'lr_model.pkl')

print("Models saved.")

# To reload later:
# vec = joblib.load('vectorizer.pkl')
# mdl = joblib.load('svm_model.pkl')

## Draft Evaluation

We evaluate the classical baselines on a fixed stratified train/validation/test split so that each model sees the same data and can be compared fairly. Hyperparameters for the SVM baseline are selected on the validation split, and the final comparison is reported only once on the held-out test set. This prevents the test set from influencing model selection.

The primary metrics are accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrices. Accuracy gives an overall summary of correct predictions, while precision, recall, and F1 provide a more detailed view of class-wise behavior. ROC-AUC captures ranking quality independent of a fixed threshold, and confusion matrices make it easier to diagnose whether a model over-predicts positive or negative sentiment. To keep the comparison practical, we also record training time, inference time, and throughput in tweets per second.

To study robustness, we evaluate each model on both the full test set and a noisy subset containing tweets with Twitter-specific artifacts such as hashtags, mentions, URLs, repeated characters, emoticons, emojis, and common slang. Comparing the performance drop between the full test set and the noisy subset gives a direct measure of how sensitive each classical model is to noisy social-media text.


## Error Analysis Method and Success Criteria

Error analysis will focus on false positives and false negatives from the held-out test set. Representative errors should be grouped into common categories such as negation, sarcasm, emoji-heavy posts, slang, abbreviations, misspellings, repeated characters, and cases where sentiment depends on context that is not explicit in the tweet. This analysis should be repeated for the noisy subset to determine whether Twitter-specific noise changes the dominant failure modes.

The Member 2 contribution will be considered complete when the notebook includes a tuned linear SVM baseline, reports accuracy, precision, recall, F1, ROC-AUC, and confusion matrices on the shared split, and compares full-test versus noisy-subset performance. The written evaluation section should explain the metrics and fairness of the comparison, and the error-analysis section should clearly describe how failure cases will be inspected and interpreted.
